# Quickstart: 4-bit Base Case Validation
**Runtime → Change runtime type → T4 GPU**

In [ ]:
# Cell 1: Setup (safe to re-run)
import os
if not os.path.exists('/content/recursive-transformers/src'):
    !git clone https://github.com/dhruvsyam123/recursive-transformers.git /content/recursive-transformers
else:
    !cd /content/recursive-transformers && git pull
%cd /content/recursive-transformers
!pip install -q jax[cuda12] equinox optax jaxtyping numpy pyyaml matplotlib einops wandb

import jax
print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

In [ ]:
# Cell 2: Build 4-bit dataset — ALL 256 pairs (exhaustive, no split)
from src.data.karatsuba_trace import KaratsubaTraceGenerator
from src.data.dataset import MultiplicationDataset, DataConfig

gen = KaratsubaTraceGenerator(base_case_bits=4)
trace = gen.generate(179, 214, 8)
print(f"179 * 214 = {trace.trace_product} (correct: {trace.verify()})\n")

# Use ALL 256 pairs for training — base case must be memorized perfectly
config = DataConfig(
    bit_widths=[4],
    algorithm='karatsuba',
    base_case_bits=4,
    train_fraction=1.0,  # all pairs for training
)
ds = MultiplicationDataset(config)
print(ds.summary())

In [ ]:
# Cell 3: Train 4-bit base case
from src.model.looped_transformer import create_model, count_parameters
from src.data.tokenizer import Tokenizer
import jax
import jax.numpy as jnp
import equinox as eqx
import optax

tok = Tokenizer()
# Token IDs for tags — tokens after [INPUT] up to the next tag are unpredictable input data
INPUT_ID = tok.encode_token("[INPUT]")
BASE_MUL_ID = tok.encode_token("[BASE_MUL]")

key = jax.random.PRNGKey(42)
model = create_model(
    key=key,
    vocab_size=143,
    d_model=128,
    n_heads=4,
    d_ff=256,
    n_shared_layers=1,
    max_loop_iterations=8,
    max_seq_len=32,
    position_encoding_type='sinusoidal',
)

N_LOOPS = 4
print(f"Model parameters: {count_parameters(model):,}")

schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=3e-3, warmup_steps=100,
    decay_steps=8000, end_value=1e-6
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.01)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

def make_predictable_mask(token_ids_full, pad_mask_full):
    """Mask out unpredictable input data bits from loss.
    Computes on full token sequence, returns mask aligned with targets (shifted by 1).

    Args:
        token_ids_full: (batch, max_len) full padded token sequence
        pad_mask_full: (batch, max_len) 1.0 for real tokens, 0.0 for padding
    Returns:
        (batch, max_len-1) mask aligned with target_ids = token_ids[:, 1:]
    """
    is_input_tag = (token_ids_full == INPUT_ID)
    is_base_tag = (token_ids_full == BASE_MUL_ID)
    in_input_region = (jnp.cumsum(is_input_tag, axis=1) > 0) & (jnp.cumsum(is_base_tag, axis=1) == 0)
    predictable = ~in_input_region | is_input_tag
    full_mask = pad_mask_full * predictable.astype(jnp.float32)
    # Shift to align with targets: target[i] = tokens[i+1]
    return full_mask[:, 1:]

@eqx.filter_jit
def train_step(model, opt_state, token_ids_full, pad_mask_full):
    input_ids = token_ids_full[:, :-1]
    target_ids = token_ids_full[:, 1:]
    mask = make_predictable_mask(token_ids_full, pad_mask_full)

    def loss_fn(model):
        def forward(ids):
            positions = jnp.arange(ids.shape[0])
            return model(ids, positions, n_loops=N_LOOPS)
        logits = jax.vmap(forward)(input_ids)
        log_probs = jax.nn.log_softmax(logits, axis=-1)
        targets_one_hot = jax.nn.one_hot(target_ids, logits.shape[-1])
        token_losses = -jnp.sum(targets_one_hot * log_probs, axis=-1)
        return jnp.sum(token_losses * mask) / jnp.maximum(jnp.sum(mask), 1.0)
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
    updates, opt_state_new = optimizer.update(
        grads, opt_state, eqx.filter(model, eqx.is_array)
    )
    model_new = eqx.apply_updates(model, updates)
    return model_new, opt_state_new, loss

print("Training 4-bit base case (4 loops, d=128)...")
for epoch in range(1000):
    epoch_loss = 0
    n_batches = 0
    for batch in ds.get_batch(split='train', batch_size=32):
        token_ids_full = jnp.array(batch['token_ids'])
        pad_mask_full = jnp.array(batch['mask'])
        model, opt_state, loss = train_step(model, opt_state, token_ids_full, pad_mask_full)
        epoch_loss += float(loss)
        n_batches += 1
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Loss: {epoch_loss/n_batches:.4f}")

print("Done!")

In [ ]:
# Cell 4: Evaluate 4-bit accuracy on ALL 256 pairs
correct = 0
total = 0
for batch in ds.get_batch(split='train', batch_size=64, shuffle=False):
    token_ids_full = jnp.array(batch['token_ids'])
    pad_mask_full = jnp.array(batch['mask'])
    input_ids = token_ids_full[:, :-1]
    target_ids = token_ids_full[:, 1:]
    eval_mask = make_predictable_mask(token_ids_full, pad_mask_full)

    def forward(ids):
        positions = jnp.arange(ids.shape[0])
        return model(ids, positions, n_loops=N_LOOPS)
    logits = jax.vmap(forward)(input_ids)
    preds = jnp.argmax(logits, axis=-1)

    matches = (preds == target_ids) | (eval_mask == 0)
    correct += int(jnp.all(matches, axis=-1).sum())
    total += len(batch['x'])

print(f"4-bit multiplication accuracy: {correct}/{total} = {correct/total:.1%}")
print()
if correct / total >= 0.99:
    print("PASS — base case learned. Ready for 8-bit Karatsuba.")
else:
    print(f"Needs more training ({total - correct} pairs wrong). Paste output back to Claude.")

---
## Phase 2: 8-bit Karatsuba Training
If 4-bit passed, run the cells below to train on 8-bit Karatsuba traces (1 level of recursion) and test length generalization to 16-bit.

In [ ]:
# Cell 6: Phase 3 — Hierarchical CONCAT positions, all bugs fixed
# This is the definitive experiment that achieved grokking:
# 14% → 45% → 80% → 90%+ accuracy on 8-bit
#
# Key design decisions:
# 1. Hierarchical encoding in CONCAT mode (not sum) — no signal dilution
# 2. num_step_types=10 (Karatsuba has 10 step types, not default 7)
# 3. Tag tokens get unique bit_significance (200+step_type, not -1→0)
# 4. Heavy weight decay (0.15) — forces algorithm learning over memorization
# 5. Randomized loop count [4,6,8] — robustness to recursion depth
# 6. Separate 4-bit/8-bit batches — no padding waste

from src.data.dataset import MultiplicationDataset, DataConfig
from src.model.position_encoding import HierarchicalPositionEncoding
from src.model.looped_transformer import SharedTransformerLayers, RMSNorm, count_parameters
from src.data.tokenizer import Tokenizer
import jax
import jax.numpy as jnp
import equinox as eqx
import optax

tok = Tokenizer()
INPUT_ID = tok.encode_token("[INPUT]")
BASE_MUL_ID = tok.encode_token("[BASE_MUL]")
SPLIT_ID = tok.encode_token("[SPLIT]")

# --- Model ---
class KaratsubaModel(eqx.Module):
    """Clean model: hierarchical concat positions + looped transformer."""
    token_embed: eqx.nn.Embedding
    hier_pos: HierarchicalPositionEncoding
    shared_block: SharedTransformerLayers
    final_norm: RMSNorm
    output_head: eqx.nn.Linear

    def __call__(self, tokens, n_loops, bit_sig, depth, sub_id, step_type):
        x = jax.vmap(self.token_embed)(tokens)
        pos = self.hier_pos(bit_sig, depth, sub_id, step_type)
        x = x + pos
        seq_len = tokens.shape[0]
        mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))
        def scan_fn(hidden, timestep):
            return self.shared_block(hidden, timestep, mask=mask), None
        x, _ = jax.lax.scan(scan_fn, x, jnp.arange(n_loops))
        x = jax.vmap(self.final_norm)(x)
        return jax.vmap(self.output_head)(x)

D_MODEL = 256
VOCAB = 143
key = jax.random.PRNGKey(42)
k1, k2, k3, k4 = jax.random.split(key, 4)

model = KaratsubaModel(
    token_embed=eqx.nn.Embedding(VOCAB, D_MODEL, key=k1),
    hier_pos=HierarchicalPositionEncoding(
        d_model=D_MODEL,
        max_bit_significance=256,
        max_recursion_depth=16,
        max_sub_problem_id=64,
        num_step_types=10,       # FIX: was 7
        combine_mode="concat",   # FIX: was "sum"
        key=k2,
    ),
    shared_block=SharedTransformerLayers(
        d_model=D_MODEL, n_heads=8, d_ff=512,
        n_layers=1, max_timesteps=20, key=k3,
    ),
    final_norm=RMSNorm(D_MODEL),
    output_head=eqx.nn.Linear(D_MODEL, VOCAB, key=k4),
)
print(f"Parameters: {count_parameters(model):,}")

# --- Datasets (separate for efficient batching) ---
ds4 = MultiplicationDataset(DataConfig(
    bit_widths=[4], algorithm='karatsuba', base_case_bits=4,
    train_fraction=1.0, seed=42,
))
ds8 = MultiplicationDataset(DataConfig(
    bit_widths=[8], algorithm='karatsuba', base_case_bits=4,
    ordering='depth_first', max_samples_per_width=4000, seed=42,
))
ml4 = ds4.get_max_seq_len('train') + 2
ml8 = ds8.get_max_seq_len('train') + 2
print(f"4-bit: {ds4.train_size()} examples, max_len={ml4}")
print(f"8-bit: {ds8.train_size()} train / {ds8.test_size()} test, max_len={ml8}")

# --- Mask ---
def make_mask(token_ids_full, pad_mask_full):
    BIT_0, BIT_1 = tok.encode_token(0), tok.encode_token(1)
    is_bit = (token_ids_full == BIT_0) | (token_ids_full == BIT_1)
    seen = jnp.cumsum((token_ids_full == SPLIT_ID) | (token_ids_full == BASE_MUL_ID), axis=1) > 0
    predictable = ~(~seen & is_bit)
    return (pad_mask_full * predictable.astype(jnp.float32))[:, 1:]

# --- Position processing: fix tag token bit_significance ---
def process_positions(pos):
    """Fix bit_significance=-1 for tag tokens: map to 200+step_type."""
    bit_sig = pos[:, :, 0]
    step_type = pos[:, :, 3]
    is_tag = bit_sig < 0
    bit_sig_fixed = jnp.where(is_tag, 200 + step_type, bit_sig)
    return bit_sig_fixed, pos[:, :, 1], pos[:, :, 2], pos[:, :, 3]

# --- Optimizer: heavy weight decay ---
schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=1e-3, warmup_steps=500,
    decay_steps=50000, end_value=1e-6
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.15)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

# --- Training step with configurable loop count ---
def make_train_step(n_loops):
    @eqx.filter_jit
    def train_step(model, opt_state, token_ids_full, pad_mask_full, positions_full):
        input_ids = token_ids_full[:, :-1]
        target_ids = token_ids_full[:, 1:]
        inp_pos = positions_full[:, :-1]
        mask = make_mask(token_ids_full, pad_mask_full)
        def loss_fn(model):
            bit_sig, depth, sub_id, step_t = process_positions(inp_pos)
            def forward_one(ids, bs, d, si, st):
                return model(ids, n_loops, bs, d, si, st)
            logits = jax.vmap(forward_one)(input_ids, bit_sig, depth, sub_id, step_t)
            log_probs = jax.nn.log_softmax(logits, axis=-1)
            targets_oh = jax.nn.one_hot(target_ids, logits.shape[-1])
            tok_loss = -jnp.sum(targets_oh * log_probs, axis=-1)
            return jnp.sum(tok_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)
        loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
        updates, new_opt = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_array))
        return eqx.apply_updates(model, updates), new_opt, loss
    return train_step

train_steps = {n: make_train_step(n) for n in [4, 6, 8]}

# --- Eval ---
def quick_eval(model, n_bits, n_loops, n_samples=200):
    ev = MultiplicationDataset(DataConfig(
        bit_widths=[n_bits], algorithm='karatsuba', base_case_bits=4,
        max_samples_per_width=n_samples, train_fraction=0.0, seed=999,
    ))
    ml = ev.get_max_seq_len('test') + 2
    correct, total = 0, 0
    for batch in ev.get_batch(split='test', batch_size=16, shuffle=False, max_len=ml):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['mask'])
        ps = jnp.array(batch['positions'])
        inp = tk[:, :-1]
        tgt = tk[:, 1:]
        inp_pos = ps[:, :-1]
        em = make_mask(tk, pm)
        bit_sig, depth, sub_id, step_t = process_positions(inp_pos)
        def fwd(ids, bs, d, si, st):
            return model(ids, n_loops, bs, d, si, st)
        logits = jax.vmap(fwd)(inp, bit_sig, depth, sub_id, step_t)
        preds = jnp.argmax(logits, axis=-1)
        matches = (preds == tgt) | (em == 0)
        correct += int(jnp.all(matches, axis=-1).sum())
        total += len(batch['x'])
    return correct, total

# --- Training loop ---
import random
rng = random.Random(42)

print(f"\nTraining (hierarchical CONCAT, weight_decay=0.15, random loops)...")
for epoch in range(500):
    epoch_loss, n_batches = 0.0, 0
    # 4-bit batches
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in ds4.get_batch(split='train', batch_size=64, max_len=ml4):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['mask'])
        ps = jnp.array(batch['positions'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, ps)
        epoch_loss += float(loss)
        n_batches += 1
    # 8-bit batches
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in ds8.get_batch(split='train', batch_size=16, max_len=ml8):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['mask'])
        ps = jnp.array(batch['positions'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, ps)
        epoch_loss += float(loss)
        n_batches += 1
    if epoch % 50 == 0:
        avg = epoch_loss / n_batches
        c8, t8 = quick_eval(model, 8, 8)
        print(f"Epoch {epoch:4d} | Loss: {avg:.4f} | 8-bit: {c8}/{t8} = {c8/t8:.1%}")

print("\nTraining complete.")
c8, t8 = quick_eval(model, 8, 8)
print(f"Final 8-bit: {c8}/{t8} = {c8/t8:.1%}")

In [ ]:
# Cell 7: Length generalization evaluation
print("=" * 50)
print("LENGTH GENERALIZATION EVALUATION")
print("Hierarchical Concat + All Fixes")
print("=" * 50)
print(f"\n{'Bits':>6} {'Loops':>6} {'Correct':>8} {'Total':>6} {'Accuracy':>10}")
print("-" * 44)

for n_bits, n_loops in [(4, 8), (8, 8), (16, 12), (32, 16)]:
    try:
        n_samples = 200 if n_bits <= 16 else 50
        c, t = quick_eval(model, n_bits, n_loops, n_samples=n_samples)
        print(f"{n_bits:6d} {n_loops:6d} {c:8d} {t:6d} {c/t:10.1%}")
    except Exception as e:
        print(f"{n_bits:6d} {n_loops:6d} {'ERROR':>8} {'':>6} {str(e)[:50]}")

print("\n" + "=" * 50)

# Save model
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
CKPT_DIR = '/content/drive/MyDrive/karatsuba_checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
eqx.tree_serialise_leaves(os.path.join(CKPT_DIR, 'model_final.eqx'), model)
print("Model saved to Drive.")

In [ ]:
# Cell 8: Save model (alternative: to Colab local disk)
import os
CKPT_DIR = '/content/checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
eqx.tree_serialise_leaves(os.path.join(CKPT_DIR, 'model_final.eqx'), model)
print(f"Model saved to {CKPT_DIR}")